In [2]:
import sys
from datetime import datetime
import pandas_ta_classic as ta

project_root = r"D:\Work\DeepVision\nexora-trading-framework"
if project_root not in sys.path:
    sys.path.append(project_root)

from config import settings
from app.brokers.factory import BrokerFactory, AvailableBrokers
from app.brokers.market_data.provider import DataFormat
from app.tradingview.smc_indicator import SMCIndicator
from app.tradingview.types import SMCConfig
from pandas_luarix import smc

In [3]:
# Available SYMBOLS: BCHUSDT, BTCUSDT, ETHUSDT, BNBUSDT, ADAUSDT, SOLUSDT, XRPUSDT, DOTUSDT, DOGEUSDT, AVAXUSDT, MATICUSDT, EURUSDT

SYMBOL = settings.PIPELINE_SYMBOL
INTERVAL = settings.PIPELINE_INTERVAL
CAPITAL = settings.CAPITAL
STOP_LOSS_PERCENT = settings.STOP_LOSS_PERCENT
STOP_LIMIT_PERCENT = settings.STOP_LIMIT_PERCENT
START = settings.START
END = datetime(2024, 1, 1)

In [4]:
binance_provider = BrokerFactory.get_broker(AvailableBrokers.BINANCE, False)

In [5]:
monthly_candles = binance_provider.market_data_provider.get_historical_candles(SYMBOL, start=START, end=END, interval=INTERVAL, format=DataFormat.OHLCV)


In [6]:
atr_precomputed = ta.atr(monthly_candles["high"], monthly_candles["low"], monthly_candles["close"], 14)
ohlcv_df = monthly_candles.reset_index(drop=True)
config   = SMCConfig(swing_length=50, equal_length=10)
result   = SMCIndicator(config).run(ohlcv_df, precomputed_atr=atr_precomputed)
#update_smc_results_equals(df=ohlcv_df, result=result, tolerance_pct=0.005)
external = [s for s in result.structure_events if not s.internal]

print(f"Total barras: {len(ohlcv_df)}  |  EQH: {len(result.equal_highs)}  EQL: {len(result.equal_lows)}")

Total barras: 2329  |  EQH: 1  EQL: 2


In [7]:
# Comparación puro-default vs puro-default: SMCIndicator(SMCConfig()) sin
# precomputed_atr (usa su ATR(200) interno) vs pandas_luarix con sus propios
# defaults (equal_length=3 -> length=3, atr_period=200) — deben coincidir exacto.
reference_result = SMCIndicator(SMCConfig()).run(ohlcv_df)

eqh_groups = ohlcv_df.smc.equal_high(include_mitigated=True)
eql_groups = ohlcv_df.smc.equal_low(include_mitigated=True)

print(f"Reference EQH: {len(reference_result.equal_highs)}  |  pandas_luarix EQH groups: {len(eqh_groups)}")
print(f"Reference EQL: {len(reference_result.equal_lows)}  |  pandas_luarix EQL groups: {len(eql_groups)}")

print()
print("Reference equal_highs[0]:")
print(reference_result.equal_highs[0])
print()
print("pandas_luarix equal_high()[0]:")
eqh_groups[0]

Reference EQH: 20  |  pandas_luarix EQH groups: 20
Reference EQL: 18  |  pandas_luarix EQL groups: 18

Reference equal_highs[0]:
EqualLevel(first_index=187, second_index=200, price=np.float64(11710.0), bullish=False, first_time=Timestamp('2018-02-20 00:00:00+0000', tz='UTC'), second_time=Timestamp('2018-03-05 00:00:00+0000', tz='UTC'), c1=date                 2018-02-20 00:00:00+00:00
open                                  11147.11
high                                  11786.01
low                                   11100.59
close                                 11200.99
volume                            48153.354288
close_time    2018-02-20 23:59:59.999000+00:00
Name: 187, dtype: object, c2=date                 2018-03-05 00:00:00+00:00
open                                   11515.0
high                                   11710.0
low                                   11415.01
close                                  11454.0
volume                            15144.231063
close_time    2018-

,date,open,high,low,close,volume,close_time,mitigated
187,2018-02-20 00:00:00+00:00,11147.11,11786.01,11100.59,11200.99,48153.354288,2018-02-20 23:59:59.999000+00:00,True
200,2018-03-05 00:00:00+00:00,11515.00,11710.00,11415.01,11454.00,15144.231063,2018-03-05 23:59:59.999000+00:00,True


In [8]:
print(reference_result.equal_lows[0])
print()
print("pandas_luarix equal_lows()[0]:")
eql_groups[0]

EqualLevel(first_index=227, second_index=232, price=np.float64(6500.0), bullish=True, first_time=Timestamp('2018-04-01 00:00:00+0000', tz='UTC'), second_time=Timestamp('2018-04-06 00:00:00+0000', tz='UTC'), c1=date                 2018-04-01 00:00:00+00:00
open                                    6922.0
high                                   7049.98
low                                     6430.0
close                                  6813.01
volume                            44071.430463
close_time    2018-04-01 23:59:59.999000+00:00
Name: 227, dtype: object, c2=date                 2018-04-06 00:00:00+00:00
open                                    6770.0
high                                    6850.0
low                                     6500.0
close                                  6601.39
volume                             27455.01192
close_time    2018-04-06 23:59:59.999000+00:00
Name: 232, dtype: object)

pandas_luarix equal_lows()[0]:


,date,open,high,low,close,volume,close_time,mitigated
227,2018-04-01 00:00:00+00:00,6922.0,7049.98,6430.0,6813.01,44071.430463,2018-04-01 23:59:59.999000+00:00,True
232,2018-04-06 00:00:00+00:00,6770.0,6850.00,6500.0,6601.39,27455.011920,2018-04-06 23:59:59.999000+00:00,True


In [10]:
import math

# Swings: comparar swing_pivot_high/low (origen del pivot) entre la referencia y pandas_luarix
ref_sh = [(i - config.swing_length, price) for i, price in enumerate(result.signals["swing_pivot_high"]) if not math.isnan(price)]
ref_sl = [(i - config.swing_length, price) for i, price in enumerate(result.signals["swing_pivot_low"]) if not math.isnan(price)]

mine_sh = ohlcv_df.smc.swing_high(length=config.swing_length, include_mitigated=True)
mine_sl = ohlcv_df.smc.swing_low(length=config.swing_length, include_mitigated=True)
mine_sh_positions = list(ohlcv_df.index.get_indexer(mine_sh.index))
mine_sl_positions = list(ohlcv_df.index.get_indexer(mine_sl.index))

ref_sh_pos = [p for p, _ in ref_sh]
ref_sl_pos = [p for p, _ in ref_sl]
print(f"Reference swing highs: {len(ref_sh_pos)}  |  pandas_luarix: {len(mine_sh_positions)}  |  match: {mine_sh_positions == ref_sh_pos}")
print(f"Reference swing lows:  {len(ref_sl_pos)}  |  pandas_luarix: {len(mine_sl_positions)}  |  match: {mine_sl_positions == ref_sl_pos}")

Reference swing highs: 17  |  pandas_luarix: 17  |  match: True
Reference swing lows:  18  |  pandas_luarix: 18  |  match: True


In [12]:
# BOS/CHoCH: comparar eventos de estructura swing (no internal) entre ambos lados
external = [s for s in result.structure_events if not s.internal]
ref_events = sorted(
    (s.pivot.bar_index, s.broken_index, s.direction.value, s.structure.value)
    for s in external
)

mine_events_df = ohlcv_df.smc.bos_choch(length=config.swing_length)
mine_positions = ohlcv_df.index.get_indexer(mine_events_df.index)
mine_events = sorted(
    zip(
        mine_positions,
        mine_events_df["direction"].tolist(),
        [("CHoCH" if t == "CHOCH" else t) for t in mine_events_df["structure_type"]],
    )
)

ref_events_no_pivot = [(pos, d, st) for (_pivot_idx, pos, d, st) in ref_events]
print(f"Reference BOS/CHoCH events: {len(ref_events_no_pivot)}  |  pandas_luarix: {len(mine_events)}  |  match: {mine_events == ref_events_no_pivot}")

Reference BOS/CHoCH events: 13  |  pandas_luarix: 13  |  match: True


In [14]:
# Order Blocks: comparar los swing_order_blocks activos (no mitigados) de la referencia
ref_obs = sorted(
    (ob.bar_index, ob.direction, round(ob.top, 4), round(ob.bottom, 4))
    for ob in result.swing_order_blocks
)

mine_obs_df = ohlcv_df.smc.order_blocks(length=config.swing_length, include_mitigated=False)
mine_positions = ohlcv_df.index.get_indexer(mine_obs_df.index)
mine_obs = sorted(
    zip(
        mine_positions,
        mine_obs_df["direction"].tolist(),
        mine_obs_df["top"].round(4).tolist(),
        mine_obs_df["bottom"].round(4).tolist(),
    )
)

print(f"Reference active order blocks: {len(ref_obs)}  |  pandas_luarix: {len(mine_obs)}  |  match: {mine_obs == ref_obs}")

Reference active order blocks: 7  |  pandas_luarix: 7  |  match: True


In [15]:
# Fair Value Gaps: la referencia necesita detect_fvg=True explícito (está apagado por default)
fvg_config = SMCConfig(swing_length=config.swing_length, detect_fvg=True)
fvg_result = SMCIndicator(fvg_config).run(ohlcv_df)

ref_fvgs = sorted(
    (fvg.right_index, fvg.direction, round(fvg.top, 4), round(fvg.bottom, 4))
    for fvg in fvg_result.fair_value_gaps
)

mine_fvgs_df = ohlcv_df.smc.fair_value_gaps(include_filled=False)
mine_positions = ohlcv_df.index.get_indexer(mine_fvgs_df.index)
mine_fvgs = sorted(
    zip(
        mine_positions,
        mine_fvgs_df["direction"].tolist(),
        mine_fvgs_df["top"].round(4).tolist(),
        mine_fvgs_df["bottom"].round(4).tolist(),
    )
)

print(f"Reference active FVGs: {len(ref_fvgs)}  |  pandas_luarix: {len(mine_fvgs)}  |  match: {mine_fvgs == ref_fvgs}")

Reference active FVGs: 38  |  pandas_luarix: 38  |  match: True
